In [24]:
import pandas as pd
import json

In [25]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [26]:
df_sleep = df_raw[(df_raw['Key'] == 'sleep') & (df_raw['Tag'] == 'daily_report')].copy()

# Parse JSON values and normalize into separate columns
sleep_data_list = [json.loads(val) for val in df_sleep['Value']]
df_sleep_normalized = pd.json_normalize(sleep_data_list)

# Unpack segment_details (list of dicts) - explode and normalize
if 'segment_details' in df_sleep_normalized.columns:
    df_sleep_exploded = df_sleep_normalized.explode('segment_details')
    segment_details_normalized = pd.json_normalize(df_sleep_exploded['segment_details'])
    df_sleep_exploded = df_sleep_exploded.drop('segment_details', axis=1).reset_index(drop=True)
    segment_details_normalized = segment_details_normalized.reset_index(drop=True)

    # Drop duplicate columns from segment_details_normalized (keep from df_sleep_exploded)
    duplicate_cols = segment_details_normalized.columns.intersection(df_sleep_exploded.columns)
    segment_details_normalized = segment_details_normalized.drop(columns=duplicate_cols)

    df_sleep_normalized = df_sleep_exploded.join(segment_details_normalized)

# Combine with original metadata (Time, Uid, etc.)
df_sleep_result = df_sleep[['Uid', 'Sid', 'Time']].reset_index(drop=True).join(df_sleep_normalized)

# Drop specified columns
columns_to_drop = ['total_turn_over', 'sleep_trace_duration', 'sleep_manually_duration', 'total_snore',
                   'breath_quality', 'total_body_move', 'total_snore_disturb', 'sleep_algorithm_version', 'Uid', 'Sid', 'timezone']
df_sleep_result = df_sleep_result.drop(columns=[col for col in columns_to_drop if col in df_sleep_result.columns])

df_sleep_result


,Time,total_duration,sleep_rem_duration,sleep_nap_duration,avg_hr,avg_spo2,sleep_light_duration,total_long_duration,sleep_score,day_sleep_evaluation,...,long_sleep_evaluation,awake_count,max_hr,sleep_deep_duration,min_hr,min_spo2,max_spo2,wake_up_time,duration,bedtime
0,1736899200,428,125,0,56,0,167,428,70,0,...,20,2,80,136,48,NaN,NaN,1736930100,428,1736902380
1,1736985600,515,119,65,54,0,226,450,77,0,...,17,1,79,105,43,NaN,NaN,1737012900,450,1736985060
2,1737072000,515,119,65,54,0,226,450,77,0,...,17,1,79,105,43,NaN,NaN,1737022140,65,1737018240
3,1737158400,496,121,0,57,0,247,496,86,0,...,6,2,99,128,47,NaN,NaN,1737100800,496,1737070020
4,1737244800,523,121,0,50,0,246,523,93,0,...,26,0,79,156,43,NaN,NaN,1737185400,523,1737154020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,1775088000,691,140,53,66,97,294,638,86,0,...,17,1,85,204,51,94.0,99.0,1769070600,638,1769032140
222,1775174400,691,140,53,66,97,294,638,86,0,...,17,1,85,204,51,94.0,99.0,1769116620,53,1769113440
223,1775260800,832,285,0,62,97,337,832,63,0,...,5,4,94,210,51,94.0,99.0,1769165760,832,1769113440
224,1775347200,644,136,0,54,97,316,644,76,0,...,19,3,82,192,43,93.0,99.0,1769331360,644,1769292060


In [27]:
# Convert Time columns from epoch to datetime
df_sleep_result['Date'] = pd.to_datetime(df_sleep_result['Time'], unit='s')

# Convert wake_up_time and bedtime to datetime, then extract time in HH:MM:SS format
df_sleep_result['wake_up_time'] = pd.to_datetime(df_sleep_result['wake_up_time'], unit='s').dt.strftime('%H:%M:%S')
df_sleep_result['bedtime'] = pd.to_datetime(df_sleep_result['bedtime'], unit='s').dt.strftime('%H:%M:%S')

# Drop the original Time column
df_sleep_result = df_sleep_result.drop(columns=['Time'])

df_sleep_result


,total_duration,sleep_rem_duration,sleep_nap_duration,avg_hr,avg_spo2,sleep_light_duration,total_long_duration,sleep_score,day_sleep_evaluation,sleep_awake_duration,...,awake_count,max_hr,sleep_deep_duration,min_hr,min_spo2,max_spo2,wake_up_time,duration,bedtime,Date
0,428,125,0,56,0,167,428,70,0,34,...,2,80,136,48,NaN,NaN,08:35:00,428,00:53:00,2025-01-15
1,515,119,65,54,0,226,450,77,0,14,...,1,79,105,43,NaN,NaN,07:35:00,450,23:51:00,2025-01-16
2,515,119,65,54,0,226,450,77,0,14,...,1,79,105,43,NaN,NaN,10:09:00,65,09:04:00,2025-01-17
3,496,121,0,57,0,247,496,86,0,17,...,2,99,128,47,NaN,NaN,08:00:00,496,23:27:00,2025-01-18
4,523,121,0,50,0,246,523,93,0,0,...,0,79,156,43,NaN,NaN,07:30:00,523,22:47:00,2025-01-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,691,140,53,66,97,294,638,86,0,3,...,1,85,204,51,94.0,99.0,08:30:00,638,21:49:00,2026-04-02
222,691,140,53,66,97,294,638,86,0,3,...,1,85,204,51,94.0,99.0,21:17:00,53,20:24:00,2026-04-03
223,832,285,0,62,97,337,832,63,0,40,...,4,94,210,51,94.0,99.0,10:56:00,832,20:24:00,2026-04-04
224,644,136,0,54,97,316,644,76,0,11,...,3,82,192,43,93.0,99.0,08:56:00,644,22:01:00,2026-04-05


In [28]:
import os

output_dir = '../data/processed_data'
output_path = os.path.join(output_dir, 'miband_sleep_processed.csv')
df_sleep_result.to_csv(output_path, index=False)